# Notebook 3: Analyze Your Own Audio Data

In this notebook you will:

1. **Upload** your own audio recordings to Google Drive
2. **Load** them into the notebook (supports `.m4a`, `.wav`, `.mp3`, and more)
3. **Visualize** each recording as a waveform and spectrogram
4. **Extract MFCC features** and **cluster** your recordings using an autoencoder

This is your chance to work with real data you collected yourself!

### Save Your Own Copy

1. **File → Save a copy in Drive**
2. Rename it with your name (e.g. `YourName_Notebook_3.ipynb`)

## Before You Start: Upload Your Audio Files

You need audio files in your Google Drive before running this notebook.

### How to upload:

1. Go to [drive.google.com](https://drive.google.com)
2. Create a folder called **`Audio`** (in the top level of My Drive)
3. Upload your audio files into that folder
   - Supported formats: `.m4a`, `.wav`, `.mp3`, `.ogg`, `.flac`
   - You can record audio on your phone and upload the files
   - Aim for **5+ files** to get interesting clustering results

Your Drive should look like:
```
My Drive/
  Audio/
    recording1.m4a
    recording2.m4a
    recording3.wav
    ...
```

## Step 1: Setup & Mount Google Drive

This cell installs dependencies and connects to your Google Drive so we can access your files.

In [ ]:
# Install dependencies
!pip install -q pydub

# Import libraries
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from pydub import AudioSegment
from IPython.display import Audio, display

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("Setup complete!")

## Step 2: Helper Functions

In [ ]:
def load_audio_file(file_path):
    """
    Load an audio file of any format using pydub, then convert to a numpy array.
    Returns:
        samples: float32 numpy array scaled to [-1, 1]
        sr: sample rate
    """
    audio = AudioSegment.from_file(file_path)
    samples = np.array(audio.get_array_of_samples())
    if audio.channels == 2:
        samples = samples.reshape((-1, 2)).mean(axis=1)
    samples = samples.astype(np.float32)
    max_val = float(2 ** (8 * audio.sample_width - 1))
    samples /= max_val
    return samples, audio.frame_rate


def plot_waveform(y, sr, title="Waveform"):
    """Plot the waveform of an audio signal."""
    plt.figure(figsize=(10, 3))
    librosa.display.waveshow(y, sr=sr)
    plt.title(title)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.show()


def plot_spectrogram(y, sr, title="Spectrogram"):
    """Plot the spectrogram of an audio signal."""
    S = np.abs(librosa.stft(y))
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.show()


class Autoencoder(nn.Module):
    """Compresses input features down to 2 dimensions, then reconstructs them."""
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded


print("Helper functions ready.")

## Step 3: Find Your Audio Files

This cell looks in your `My Drive/Audio` folder and lists what it finds.

**If no files are found**, double-check that:
- You created a folder called exactly `Audio` in the top level of My Drive
- You uploaded audio files into it

In [ ]:
# Change this path if your files are in a different folder
AUDIO_FOLDER = '/content/drive/MyDrive/Audio'

# Find all audio files
AUDIO_EXTENSIONS = ('*.m4a', '*.wav', '*.mp3', '*.ogg', '*.flac')
audio_files = []
for ext in AUDIO_EXTENSIONS:
    audio_files.extend(glob.glob(os.path.join(AUDIO_FOLDER, ext)))
audio_files = sorted(audio_files)

print(f"Found {len(audio_files)} audio files in {AUDIO_FOLDER}:\n")
for i, f in enumerate(audio_files, 1):
    print(f"  {i}. {os.path.basename(f)}")

## Step 4: Visualize & Listen to Each File

For each audio file, we'll display the waveform, spectrogram, and an audio player.

In [ ]:
for file_path in audio_files:
    name = os.path.basename(file_path)
    print(f"\n{'='*60}")
    print(f"File: {name}")
    print(f"{'='*60}")

    try:
        y, sr = load_audio_file(file_path)
        print(f"  Sample rate: {sr} Hz  |  Duration: {len(y)/sr:.2f}s")

        plot_waveform(y, sr, title=f"{name} — Waveform")
        plot_spectrogram(y, sr, title=f"{name} — Spectrogram")

        print("  Click play to listen:")
        display(Audio(y, rate=sr))
    except Exception as e:
        print(f"  Error processing {name}: {e}")

## Step 5: Extract MFCC Features

MFCCs (Mel Frequency Cepstral Coefficients) capture the "shape" of each sound — what makes one recording sound different from another.

We extract 13 MFCCs from each file and summarize them as a mean and standard deviation, giving us 26 features per recording.

In [ ]:
N_MFCC = 13
features = []
file_names = []

for file_path in audio_files:
    try:
        y, sr = load_audio_file(file_path)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        feature_vec = np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])
        features.append(feature_vec)
        file_names.append(os.path.basename(file_path))
    except Exception as e:
        print(f"Error extracting features from {os.path.basename(file_path)}: {e}")

X = np.array(features)
print(f"Extracted features from {len(features)} files.")
print(f"Feature matrix shape: {X.shape}  (files x features)")

## Step 6: Train the Autoencoder & Cluster

We train an autoencoder to compress the 26 MFCC features down to just **2 dimensions** so we can plot them on a scatter plot.

Recordings that sound similar should appear close together on the plot.

In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

# Train the autoencoder
model = Autoencoder(input_dim=X_scaled.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

NUM_EPOCHS = 2000
model.train()
for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    encoded, decoded = model(X_tensor)
    loss = criterion(decoded, X_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}  —  Loss: {loss.item():.4f}")

print("Training complete!")

In [ ]:
# Generate 2D embeddings
model.eval()
with torch.no_grad():
    embeddings, _ = model(X_tensor)
embeddings = embeddings.numpy()

# Plot each recording as a labeled point
plt.figure(figsize=(10, 7))
plt.scatter(embeddings[:, 0], embeddings[:, 1], s=80, alpha=0.7)

# Label each point with its filename
for i, name in enumerate(file_names):
    plt.annotate(name, (embeddings[i, 0], embeddings[i, 1]),
                 fontsize=8, ha='left', va='bottom')

plt.title("2D Embedding of Your Audio Files (Autoencoder on MFCC Features)", fontsize=13)
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Recordings that sound similar should appear close together!")
print("Try adding more files to your Audio folder and re-running to see how the clusters change.")

## Bonus: Color by Group

If you organized your files so that related recordings are named similarly (e.g. `bird1_a.m4a`, `bird1_b.m4a`, `bird2_a.m4a`), you can assign groups.

The cell below groups every 5 files together (based on alphabetical order). Edit `GROUP_SIZE` to match your data.

In [ ]:
GROUP_SIZE = 5  # Change this to match your data

# Assign group labels
group_labels = [chr(ord('A') + i // GROUP_SIZE) for i in range(len(file_names))]

plt.figure(figsize=(10, 7))
for label in sorted(set(group_labels)):
    mask = [g == label for g in group_labels]
    pts = embeddings[mask]
    plt.scatter(pts[:, 0], pts[:, 1], label=f"Group {label}", s=80, alpha=0.7)

plt.legend(fontsize=11)
plt.title("2D Embeddings Colored by Group", fontsize=13)
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

**Experiment ideas:**
- Record different people speaking and see if the autoencoder separates them
- Record the same bird at different times and see how consistent the songs are
- Mix bird recordings with non-bird sounds — can the autoencoder tell them apart?